# Iceberg Tables

In this notebook you will create a simple Iceberg table from the `answers` dataset and explore the most useful features that Iceberg adds on top of plain Parquet.

Tasks:
1. Configure the Spark session with an Iceberg catalog (Hadoop catalog in a local warehouse).
2. Read the `answers` dataset (Parquet) into a DataFrame.
3. Create an Iceberg table from the DataFrame.
4. Verify the table by reading it back and listing the catalog content.
5. Inspect Iceberg metadata (snapshots and data files).
6. Schema evolution: add a column.
7. Schema evolution: rename a column.
8. Schema evolution: drop a column.
9. Append more rows and observe a new snapshot.
10. Time travel: query an earlier snapshot of the table.
11. Create a partitioned Iceberg table (hidden partitioning).
12. Partition evolution: change the partition spec without rewriting data.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

import os

#### Paths:

* `answers_input_path` points to the existing Parquet dataset shipped with the training.
* `warehouse_path` is where Iceberg will store the data and metadata files of the new tables.

In [ ]:
base_path = os.getcwd()

project_path = ('/').join(base_path.split('/')[0:-3]) 

answers_input_path = os.path.join(project_path, 'data/answers')

warehouse_path = os.path.join(project_path, 'output/iceberg-warehouse')

### Task 1: Configure the Spark session with an Iceberg catalog

Hint:
* register the Iceberg SQL extensions: `spark.sql.extensions = org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions`
* define a catalog named `local` of type `hadoop` pointing to the warehouse folder:
  * `spark.sql.catalog.local = org.apache.iceberg.spark.SparkCatalog`
  * `spark.sql.catalog.local.type = hadoop`
  * `spark.sql.catalog.local.warehouse = <warehouse_path>`
* the Iceberg runtime jar is already on the classpath because it was placed in `$SPARK_HOME/jars/` during the installation
* docs for [Iceberg Spark configuration](https://iceberg.apache.org/docs/latest/spark-configuration/)

In [ ]:
spark = (
    SparkSession
    .builder
    .appName('Iceberg Tables')
    .config('spark.sql.extensions', 'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions')
    .config('spark.sql.catalog.local', 'org.apache.iceberg.spark.SparkCatalog')
    .config('spark.sql.catalog.local.type', 'hadoop')
    .config('spark.sql.catalog.local.warehouse', warehouse_path)
    .getOrCreate()
)

In [ ]:
# Silence the benign FileNotFoundException warning the Hadoop catalog
# emits when it probes for version-hint.text on a brand-new table.
spark.sparkContext._jvm.org.apache.logging.log4j.core.config.Configurator.setLevel(
    'org.apache.iceberg.hadoop.HadoopTableOperations',
    spark.sparkContext._jvm.org.apache.logging.log4j.Level.ERROR,
)

In [ ]:
print(spark.version)

### Task 2: Read the answers dataset

Hint:
* the dataset is in Parquet format
* check the schema and the first records

In [ ]:
answersDF = (
    spark.read.parquet(answers_input_path)
)

answersDF.printSchema()

answersDF.show(n=5, truncate=20)

### Task 3: Create an Iceberg table from the DataFrame

Create a namespace (database) `db` in the `local` catalog and write the DataFrame as an Iceberg table named `answers`. The fully qualified table name is `local.db.answers`.

Hint:
* create the namespace with SQL: `CREATE NAMESPACE IF NOT EXISTS local.db`
* use the v2 writer with `using('iceberg')` and `createOrReplace()`
* alternatively the same can be done with SQL: `CREATE TABLE local.db.answers USING iceberg AS SELECT * FROM ...`


In [ ]:
spark.sql('CREATE NAMESPACE IF NOT EXISTS local.db')

In [ ]:
(
    answersDF
    .writeTo('local.db.answers')
    .using('iceberg')
    .createOrReplace()
)

### Task 4: Verify the table

Hint:
* list the tables in the namespace: `SHOW TABLES IN local.db`
* read the table back with `spark.table('local.db.answers')` (or `SELECT * FROM local.db.answers`)
* check the row count and a few records

In [ ]:
spark.sql('SHOW TABLES IN local.db').show()

In [ ]:
iceberg_answersDF = spark.table('local.db.answers')

iceberg_answersDF.printSchema()

print('row count:', iceberg_answersDF.count())

iceberg_answersDF.show(n=5, truncate=20)

### Task 5: Inspect Iceberg metadata

Iceberg exposes the metadata of a table as virtual sub-tables. Inspect them to see what is happening under the hood after the initial write.

Hint:
* `local.db.answers.snapshots` — one entry per commit
* `local.db.answers.files` — the data files registered in the current snapshot
* `local.db.answers.history` — full history of snapshots
* docs for [Iceberg metadata tables](https://iceberg.apache.org/docs/latest/spark-queries/#inspecting-tables)

In [ ]:
spark.sql('SELECT * FROM local.db.answers.snapshots').show(truncate=30)

In [ ]:
spark.sql('SELECT file_path, record_count, file_size_in_bytes FROM local.db.answers.files').show(truncate=False)

In [ ]:
spark.sql('SELECT * FROM local.db.answers.history').show(truncate=False)

### Task 6: Schema evolution — add a column

Iceberg supports full schema evolution (add, rename, drop, reorder, change type) as a metadata-only change — no data files are rewritten.

Add a new boolean column `is_archived` to the table and verify the new schema.

Hint:
* `ALTER TABLE local.db.answers ADD COLUMN is_archived boolean`
* docs for [Iceberg ALTER TABLE](https://iceberg.apache.org/docs/latest/spark-ddl/#alter-table)

In [ ]:
spark.sql('ALTER TABLE local.db.answers ADD COLUMN is_archived boolean')

spark.table('local.db.answers').printSchema()

### Task 7: Schema evolution — rename a column

Rename the `score` column to `points`.

Hint:
* `ALTER TABLE local.db.answers RENAME COLUMN score TO points`
* the underlying data files are not touched — Iceberg uses field IDs internally so a rename is purely metadata

In [ ]:
spark.sql('ALTER TABLE local.db.answers RENAME COLUMN score TO points')

spark.table('local.db.answers').printSchema()

### Task 8: Schema evolution — drop a column

Drop the `is_archived` column we just added.

Hint:
* `ALTER TABLE local.db.answers DROP COLUMN is_archived`

In [ ]:
spark.sql('ALTER TABLE local.db.answers DROP COLUMN is_archived')

spark.table('local.db.answers').printSchema()

### Task 9: Append more rows and observe a new snapshot

Every commit (append, overwrite, schema change, ...) creates a new snapshot. Capture the current snapshot id first — we will use it for time travel in the next task — and then append a few synthetic rows.

Hint:
* read the latest snapshot id from `local.db.answers.snapshots`
* build a small DataFrame from the existing table (e.g. take 3 rows and flip the sign of `answer_id` so they don't collide) and call `.writeTo('local.db.answers').append()`
* re-query `local.db.answers.snapshots` and confirm a second entry has appeared

In [ ]:
first_snapshot_id = (
    spark.sql('SELECT snapshot_id FROM local.db.answers.snapshots ORDER BY committed_at ASC LIMIT 1')
    .collect()[0][0]
)

print('first snapshot id:', first_snapshot_id)

In [ ]:
new_answers = (
    spark.table('local.db.answers')
    .limit(3)
    .withColumn('answer_id', col('answer_id') * -1)
)

new_answers.writeTo('local.db.answers').append()

In [ ]:
spark.sql('SELECT snapshot_id, parent_id, operation, committed_at FROM local.db.answers.snapshots ORDER BY committed_at').show(truncate=False)

### Task 10: Time travel

Query the table as it was at the first snapshot and verify it has fewer rows than the current state.

Hint:
* in SQL: `SELECT count(*) FROM local.db.answers VERSION AS OF <snapshot_id>`
* equivalently in the DataFrame API: `spark.read.option('snapshot-id', <snapshot_id>).format('iceberg').load('local.db.answers')`
* you can also travel by timestamp: `TIMESTAMP AS OF '2026-...'`
* docs for [time travel](https://iceberg.apache.org/docs/latest/spark-queries/#time-travel)

In [ ]:
current_count = spark.table('local.db.answers').count()

old_count = spark.sql(
    f'SELECT count(*) AS cnt FROM local.db.answers VERSION AS OF {first_snapshot_id}'
).collect()[0]['cnt']

print('current count:    ', current_count)
print('count at snapshot:', old_count)

### Task 11: Create a partitioned Iceberg table

Iceberg supports *hidden partitioning*: the partition value is computed from a column via a transform (`bucket`, `truncate`, `years`, `months`, `days`, `hours`, identity). Queries simply filter by the source column and Iceberg figures out which partitions to prune — there is no separate partition column to maintain.

Create a new table `local.db.answers_partitioned` from the current answers table, partitioned by `years(creation_date)`.

Hint:
* SQL: `CREATE TABLE local.db.answers_partitioned USING iceberg PARTITIONED BY (years(creation_date)) AS SELECT * FROM local.db.answers`
* check the produced layout via the `partitions` and `files` metadata tables
* docs for [partitioning](https://iceberg.apache.org/docs/latest/partitioning/

In [ ]:
spark.sql('DROP TABLE IF EXISTS local.db.answers_partitioned')

spark.sql('''
    CREATE TABLE local.db.answers_partitioned
    USING iceberg
    PARTITIONED BY (years(creation_date))
    AS SELECT * FROM local.db.answers
''')

In [ ]:
spark.sql('SELECT partition, record_count, file_count FROM local.db.answers_partitioned.partitions').show(truncate=False)

In [ ]:
spark.stop()